In [1]:
import os
import re
import sys
if '/mnt/d/HLS Kelp Detection/tools' not in sys.path:
    sys.path.append('/mnt/d/HLS Kelp Detection/tools')
import kelp_tools_linux as kt
import time 
import rasterio
import cupy as cp
import cudf
import numpy as np
from rasterio.errors import RasterioIOError
import matplotlib.pyplot as plt
from cuml.ensemble import RandomForestClassifier as cuRF
from cuml.model_selection import train_test_split
from scipy.stats import randint
import pickle
import csv
from cupyx.scipy.ndimage import binary_dilation, convolve
from IPython.display import clear_output
from matplotlib.colors import ListedColormap
import filing_tools as ft
from pyproj import Transformer
from matplotlib import rcParams
from IPython.display import clear_output

In [2]:
reclassify = False #Reclassify previously classified images
show_image = False
sleep = False
#classified_path = r'/mnt/d/hls_kelp/imagery/rf_classified_cuML'
save_final_data = True
version = 1
cloud_cover_threshold = .95
save_EMs = False
use_constant_EM = False
only_pairs = False
backup_frequency = 5
save_to_path = rf'/mnt/d/HLS Kelp Detection/processed imagery/tiles'
tiles = os.listdir('/mnt/d/HLS Kelp Detection/imagery/tiles')
tiles.remove('cloudy_images.txt')

#H:\HLS_data\imagery\Catalina\11SLT

rf_model = 'cu_rf10'
rf_path = os.path.join(r'/mnt/d/HLS Kelp Detection/random_forest',rf_model)
num_iterations = 200000

cloud_list_path = '/mnt/d/HLS Kelp Detection/imagery/tiles/cloudy_images.txt'
#unclassified_path = r'/mnt/d/hls_kelp/imagery/rf_prepped_v2'
#unclassified_files = os.listdir(unclassified_path)

In [3]:
def reduce_cudf_zero(df):
    arr = df.to_cupy()
    non_zero_mask = cp.any(arr != 0, axis=1)
    reduced_df = df[non_zero_mask]
    original_indices = cp.arange(arr.shape[0])
    filtered_indices = original_indices[non_zero_mask]
    return reduced_df, filtered_indices, original_indices 
def revert_cudf_zero(reduced_df, filtered_indices, original_indices):
    # Create a full-sized column initialized with zeros or NaNs
    # if reduced_df.ndim == 1:  # Handle 1D case (single column)
    full_df = cudf.Series(cp.full(len(original_indices), 3))
    # else:  # Handle 2D case (multiple columns)
    #     full_df = cudf.DataFrame(cp.full((len(original_indices), reduced_df.shape[1]), cp.nan))
    # Insert the reduced data into the appropriate positions
    full_df.iloc[filtered_indices] = reduced_df
    
    return full_df

In [ ]:

with open(cloud_list_path, 'r') as file:
    cloud_list = [(line.strip()) for line in file]
print(cloud_list)


for tile in tiles:
    endmember_path = rf'/mnt/d/HLS_Kelp/python_objects/EM_Dict_v{version}.pkl'
    path = os.path.join(f'/mnt/d/HLS Kelp Detection/imagery/tiles/{tile}')
    dem_path = os.path.join(path,'dem')
    if only_pairs:
        all_files  = ft.get_paired_filenames(path)
    else:
        all_files = os.listdir(path)
    all_files.remove('dem')
    for item in all_files:
        if os.path.isdir(os.path.join(path, item)):
            hls_path = os.path.join(path, item)
            granule = item
            break
        else:
            continue

    pattern = re.compile(rf'B02.tif$')
    files = os.listdir(os.path.join(path, item))
    img_files = [f for f in files if re.search(pattern, f)]
    geotiff_path = os.path.join(hls_path, img_files[0])
    try:
        land_mask = kt.create_land_mask(hls_path=geotiff_path, dem_path=dem_path, show_image=False, as_numpy=True)
    except:
        continue
    del files, img_files, pattern

    with open(rf_path, 'rb') as f:
        cu_rf = pickle.load(f)

    iterations = 0

    if (save_final_data) and not os.path.isdir(save_to_path):
        os.mkdir(save_to_path)


    endmember_dict = {}
    for item in all_files:
        if item in cloud_list:
            continue
        start_time = time.time()
        #print_memory_usage("Before processing granule")

        if not reclassify and os.path.isfile(os.path.join(save_to_path,tile, f'{item}.tif')):
            #print(f'{item} already processed. Skipping.')
            continue

        print(f'Starting {item}')
        if iterations > num_iterations:
            break

        # Define image path
        if os.path.isdir(os.path.join(path, item)):
            img_path = os.path.join(path, item)
        else:
            print('object is file, continue')
            continue
        
        try:
            sorted_files = kt.filter_and_sort_files(img_path, item)
        except:
            print(f"{item} failed to sort filenames, skipping")
            continue

        if len(sorted_files) != 6:
            print(f'Incomplete file download: {item}')
            continue

        metadata = kt.get_metadata(img_path)
        if metadata is None:
            print(f"{item} missing metadata. Skipping")
            continue
        try:
            cloud_land_mask, cloud_but_not_land_mask, percent_cloud_covered = kt.create_qa_mask(land_mask, img_path)
        except:
            continue
        if percent_cloud_covered >= cloud_cover_threshold:
            #print(f'{item} Percent cloud covered: {percent_cloud_covered}')
            del cloud_land_mask, cloud_but_not_land_mask
            continue
        print(f'{item} Percent cloud covered: {percent_cloud_covered}')
        img_bands = []
        crs = None
        transform = None
        pred_time = time.time()
        try:
            cp_stream = cp.cuda.Stream(non_blocking=True)  # Stream for asynchronous CuPy operations
            with cp_stream:
                with rasterio.Env():  # Ensuring Rasterio is working with the environment
                    for file in sorted_files:
                        with rasterio.open(os.path.join(img_path, file)) as src:
                            # Read all bands at once if possible to reduce I/O operations
                            band = cp.asarray(src.read(1))  # Read into a CuPy array
                            img_bands.append(cp.where(cloud_land_mask, 0, band))  # Mask operation
                            
                            # Initialize transform and CRS only once
                            if transform is None:
                                transform = src.transform
                                crs = src.crs

            del cloud_land_mask
        except RasterioIOError as e:
            print(f"Error reading file {file} in granule {item}: {e}")
            continue 

        print(f'Files Loaded. Duration:{time.time() - pred_time} seconds')

        pred_time= time.time()
        img = cp.stack(img_bands, axis=0)
        del img_bands
        n_bands, height, width = img.shape

        img_2D_normalized = kt.normalize_img(img, flatten=False)
        img_data = cudf.DataFrame(img_2D_normalized).astype(np.float32)
        
        del img_2D_normalized
        
        print(f'bands normalized for RF. Duration:{time.time()-pred_time} seconds')
        
        pred_time = time.time()

        reduced_img_data, filtered_indices, original_indices = reduce_cudf_zero(img_data)

        del img_data
        kelp_pred_shrunk= cu_rf.predict(reduced_img_data).astype(cp.float32)

        kelp_pred = revert_cudf_zero(kelp_pred_shrunk.astype(cp.uint8), filtered_indices, original_indices)

        #kelp_pred_cp = cp.asarray(kelp_pred)
        #kelp_pred = cp.where(cp.isnan(kelp_pred_cp), 3, kelp_pred)
        del reduced_img_data, filtered_indices,original_indices
        print(f'RF finished. Duration:{time.time()-pred_time} seconds')


        classified_img = kelp_pred.values_host.reshape(width, height)

        del kelp_pred
        classified_img = cp.where(cloud_but_not_land_mask, 2, cp.asarray(classified_img)) #
        if show_image:
            plt.figure(figsize=(25, 25)) 
            plt.subplot(2, 1, 1)  
            plt.imshow(cp.asnumpy(classified_img))
            plt.colorbar()
            plt.title(file)
            r_nor = img[2, :, :].reshape((height, width))
            g_nor = img[1, :, :].reshape((height, width))
            b_nor = img[0, :, :].reshape((height, width))
            rgb_nor_gpu = cp.stack([r_nor, g_nor, b_nor], axis=-1) 
            rgb_nor = cp.asnumpy(rgb_nor_gpu)
            rgb_cropped = cp.asnumpy(rgb_nor)#[2500:3000,500:1500]#[2700:3400, 600:2000])
            plt.figure(figsize=(15,15))
            plt.imshow(rgb_cropped)
            plt.title("RGB Cropped Image")
            plt.show()
            del rgb_cropped, rgb_nor_gpu, rgb_nor

        mesma_mask_params = [
            100,      # ocean_dilation_size
            6,        # kelp_neighborhood
            3,        # min_kelp_count
            5,       # kelp_dilation_size
            15,       # variance_window_size
            0.95,      # variance_threshold
            True #Variance Mask 
        ]
        pred_time=time.time()
        kelp_mask, ocean_mask = kt.create_mesma_mask(classified_img, img, land_mask, cloud_but_not_land_mask, *mesma_mask_params)
        print(f'Masking finished. Duration:{time.time()-pred_time} seconds')
        img = cp.asnumpy(img)
        classified_img_cpu = cp.asnumpy(classified_img)
        del cloud_but_not_land_mask, classified_img
        ocean_EM = kt.select_ocean_endmembers(ocean_mask=ocean_mask, print_average=False)
        if ocean_EM is None:
            with open(cloud_list_path, 'a') as file:
                    file.write(f"{item}\n")
            cloud_list.append(file)
            print(cloud_list)
            continue

        pred_time = time.time()
        mesma, minVals = kt.run_mesma(kelp_mask, ocean_EM, print_status=False)
        print(f'MESMA finished. Duration:{time.time()-pred_time} seconds')
        if(save_EMs):
            endmember_dict[item] = cp.asnumpy(ocean_EM)
        del ocean_EM, ocean_mask
        minVals = cp.where(cp.isnan(minVals),-999,minVals)
        min_vals = cp.asnumpy(minVals)
        del minVals
        #mesma = cp.where(cp.isnan(mesma), 0, mesma).astype(cp.int16)

        mesma_array = cp.asnumpy(mesma)
        del mesma

        if show_image:
            kelp_img = cp.asnumpy(kelp_mask).astype(np.float32)
            Mes_array_vis = np.where(mesma_array <5 , np.nan, mesma_array)
            #Mes_array_vis = np.where(Mes_array_vis >70 , np.nan, Mes_array_vis)
            kelp_vis = np.where(kelp_img == 0, np.nan, kelp_img)
            plt.figure(figsize=(12, 6), dpi=400)
            #plt.imshow(kelp_img[1], cmap='Greys', alpha=1, extent=extent_latlon, vmax=600) #,2800:3250, 875:1300]
            plt.imshow(Mes_array_vis, alpha=1)#, vmax=80) #[2800:3250, 875:1300][2500:3000,500:1500]
            cbar = plt.colorbar( shrink=.75)
            cbar.ax.tick_params(labelsize=16)  # Set the font size for colorbar ticks
            cbar.set_label('Kelp Endmember Number Multiple', fontsize=20)
            plt.xlabel('Longitude', fontsize=30)
            plt.ylabel('Latitude', fontsize=30)
            #plt.axis('off')
            plt.title("MESMA Output", fontsize=36)
            plt.xticks([],fontsize=16)
            plt.yticks([],fontsize=16)
            plt.show()
            del kelp_mask
        pred_time=time.time()
        if save_final_data:
            num_bands = 3
            data_type = rasterio.int16
            profile = {
                'driver': 'GTiff',
                'width': width,
                'height': height,
                'count': num_bands,  # one band  B02, B03, B04, and B05, classified, mesma (Blue, Green, Red, and NIR).
                'dtype': data_type,  # assuming binary mask, adjust dtype if needed
                'crs': src.crs,
                'transform': src.transform,
                'nodata': 0,  # assuming no data is 0
                'tags': {'TIMESTAMP': metadata['SENSING_TIME'], 'CLOUD_COVERAGE': percent_cloud_covered, 'RF_MODEL': rf_model, 'VIS_LINK': metadata['data_vis_url']}
            }
            if not os.path.isdir(os.path.join(save_to_path, tile)):
                os.mkdir(os.path.join(save_to_path, tile))
            img_path = os.path.join(save_to_path, tile, f'{item}.tif')

            # Write the land mask array to geotiff
            with rasterio.open(img_path, 'w', **profile) as dst:
                dst.write(classified_img_cpu.astype(np.uint8), 1)
                dst.write(mesma_array.astype(data_type), 2)
                dst.write(min_vals.astype(np.float16), 3)
                dst.update_tags(TIMESTAMP=metadata['SENSING_TIME'], CLOUD_COVERAGE=percent_cloud_covered, RF_MODEL=rf_model, VIS_LINK=metadata['data_vis_url'])

        iterations += 1
        print(f'Processed Data Saved. Duration:{time.time()-pred_time} seconds')
        clear_output(wait=True)
        print(f"File complete: {item} | Iteration: {iterations} | Time: {time.time() - start_time}")

        del img, classified_img_cpu, mesma_array, min_vals
        
        if save_EMs and iterations % backup_frequency == 0:
            if os.path.isfile(endmember_path):
                with open(endmember_path, 'rb') as f:
                    endmember_log = pickle.load(f)
            else:
                endmember_log = []
            endmember_log.append(endmember_dict)
            
            with open(endmember_path, 'wb') as f:
                pickle.dump(endmember_log, f)
            endmember_dict = {}
            print(f'Endmembers Logged on iteration: {iterations}')


        if sleep and iterations % 30 == 0 and iterations != 0:
            print("Cooling down GPU...")
            time.sleep(240)
  
                
        if os.path.isfile(endmember_path):
            with open(endmember_path, 'rb') as f:
                endmember_log = pickle.load(f)
        else:
            endmember_log = []
        endmember_log.append(endmember_dict)

        with open(endmember_path, 'wb') as f:
            pickle.dump(endmember_log, f)
    del land_mask
        # print(full_df.shape)
        # print(filtered_indices.shape)
        # print(reduced_df.shape)

['HLS.S30.T10TCN.2017251T190919.v2.0', 'HLS.L30.T19QFV.2023237T145006.v2.0', 'HLS.S30.T10TCN.2019321T191659.v2.0', 'HLS.S30.T10UCU.2018134T191911.v2.0', 'HLS.S30.T10UCU.2018341T191751.v2.0', 'HLS.S30.T10UCU.2019246T190911.v2.0', 'HLS.S30.T10UCU.2020341T191801.v2.0', 'HLS.S30.T10UCU.2021205T190921.v2.0', 'HLS.S30.T10UCU.2022208T191919.v2.0']

Starting HLS.L30.T10SGC.2017039T183429.v2.0
Incomplete file download: HLS.L30.T10SGC.2017039T183429.v2.0
Starting HLS.L30.T10SGC.2017183T183417.v2.0
Starting HLS.L30.T10SGC.2017199T183422.v2.0
Starting HLS.L30.T10SGC.2017215T183429.v2.0
HLS.L30.T10SGC.2017215T183429.v2.0 Percent cloud covered: 0.11112061646321628
Error reading file HLS.L30.T10SGC.2017215T183429.v2.0.B02.tif in granule HLS.L30.T10SGC.2017215T183429.v2.0: Read or write failed. /mnt/d/HLS Kelp Detection/imagery/tiles/10SGC/HLS.L30.T10SGC.2017215T183429.v2.0/HLS.L30.T10SGC.2017215T183429.v2.0.B02.tif, band 1: IReadBlock failed at X offset 6, Y offset 0: TIFFReadEncodedTile() failed.
St

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f4b479c5b90>>
Traceback (most recent call last):
  File "/home/atticus/miniconda3/envs/hls-linux2/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


In [7]:
if os.path.isfile(endmember_path):
    with open(endmember_path, 'rb') as f:
        endmember_log = pickle.load(f)
else:
    endmember_log = []
endmember_log.append(endmember_dict)

with open(endmember_path, 'wb') as f:
    pickle.dump(endmember_log, f)